# Part 1 - Data Cleaning: Retail Orders

**Student:** V S Praveen Ganapathiraju (ID 1234)
**Repository:** `vspraveenganapathiraju_1234_part1_data_cleaning`

## Business problem
A retail analytics team exported ~930 order records (`raw_orders.xlsx`) from several source
systems. The export is messy: inconsistent text casing/spacing, four different date formats,
missing values, malformed discounts, duplicate rows, and arithmetic that does not always tie
out. Before any KPI or profitability analysis can be trusted, the data must be **cleaned,
validated, and documented** according to the rules supplied in the `business_rules` sheet.

## Objective
Produce a clean, analysis-ready dataset (`cleaned_orders.xlsx`) plus a transparent **data
quality report**, **calculated fields**, and a **cleaning log**, honouring every business rule.

In [1]:
import pandas as pd, numpy as np, re, os, json
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

pd.set_option("display.width", 160)
RAW  = "../data/raw_orders.xlsx"
DATA = "../data"; OUT = "../outputs"; SHOTS = "../screenshots"
for d in (OUT, SHOTS): os.makedirs(d, exist_ok=True)

raw   = pd.read_excel(RAW, sheet_name="raw_orders")
rules = pd.read_excel(RAW, sheet_name="business_rules")
print("Raw shape:", raw.shape)
rules

Raw shape: (932, 21)


,Rule Area,Student-Facing Rule
0,Raw file,Do not overwrite raw_orders.xlsx. Create clean...
1,Missing region,Fill as Unknown and flag in quality report.
2,Missing ship_mode,Fill as Unknown and flag in quality report.
3,Missing discount,Treat as 0 only if all other sales fields are ...
4,Discount,Flag negative discounts and unusually high dis...
5,Cancelled/Failed/Refunded,Do not include non-completed/failed/refunded r...
6,Date sequence,Flag ship_date earlier than order_date.
7,Duplicates,Remove exact duplicate copies; flag conflictin...
8,Calculated fields,"Create calculated_sales, calculated_profit, pr..."
9,Documentation,Document cleaning decisions and assumptions in...


## Step 0 - Profile the raw data

In [2]:
profile = pd.DataFrame({
    "dtype":       raw.dtypes.astype(str),
    "missing":     raw.isna().sum(),
    "missing_pct": (raw.isna().mean()*100).round(1),
    "n_unique":    raw.nunique(),
})
print("Exact duplicate rows :", raw.duplicated().sum())
print("Duplicate order_ids  :", raw["order_id"].duplicated().sum())
profile

Exact duplicate rows : 20
Duplicate order_ids  : 32


,dtype,missing,missing_pct,n_unique
order_id,str,0,0.0,900
order_date,str,0,0.0,771
ship_date,str,0,0.0,767
customer_id,str,0,0.0,855
customer_name,str,0,0.0,27
segment,str,0,0.0,16
region,str,26,2.8,15
state,str,0,0.0,19
city,str,0,0.0,20
category,str,0,0.0,13


## Step 1 - Standardise text (casing & whitespace)

In [3]:
df = raw.copy()
CATEGORICAL = ["segment","region","ship_mode","category","sub_category",
               "state","city","payment_status","order_status"]
TEXT_STRIP  = ["order_id","customer_id","customer_name","product_name"]

def norm_space(s):
    if pd.isna(s): return s
    return re.sub(r"\s+", " ", str(s).strip())

for c in TEXT_STRIP:
    df[c] = df[c].map(norm_space)
for c in CATEGORICAL:
    df[c] = df[c].map(norm_space).map(lambda x: x.title() if isinstance(x, str) else x)

before = {c: raw[c].nunique(dropna=True) for c in CATEGORICAL}
after  = {c: df[c].nunique(dropna=True)  for c in CATEGORICAL}
pd.DataFrame({"raw_distinct": before, "cleaned_distinct": after})

,raw_distinct,cleaned_distinct
segment,16,4
region,15,4
ship_mode,11,4
category,13,3
sub_category,23,13
state,19,19
city,20,20
payment_status,8,4
order_status,10,3


## Step 2 - Parse the four date formats
**Assumption:** dash dates are DD-MM-YYYY, slash dates are MM/DD/YYYY.

In [4]:
def parse_date(x):
    if pd.isna(x): return pd.NaT
    s = str(x).strip()
    for fmt in ("%Y-%m-%d", "%d %b %Y", "%d-%m-%Y", "%m/%d/%Y"):
        try: return pd.to_datetime(s, format=fmt)
        except (ValueError, TypeError): continue
    return pd.to_datetime(s, errors="coerce")

df["order_date"] = df["order_date"].map(parse_date)
df["ship_date"]  = df["ship_date"].map(parse_date)
print("Unparsed order_date:", df["order_date"].isna().sum())
print("Unparsed ship_date :", df["ship_date"].isna().sum())
print("Order date range   :", df["order_date"].min().date(), "->", df["order_date"].max().date())

Unparsed order_date: 0
Unparsed ship_date : 0
Order date range   : 2024-01-01 -> 2025-10-12


## Step 3 - Clean & validate discount

In [5]:
def clean_discount(x):
    if pd.isna(x): return np.nan
    s = str(x).strip()
    if s.endswith("%"): return round(float(s[:-1]) / 100, 4)
    return float(s)

df["discount"] = df["discount"].map(clean_discount)
VALID_MAX = 0.65

flag_disc_neg  = df["discount"] < 0
flag_disc_high = df["discount"] > VALID_MAX
sales_fields_ok = (df[["quantity","unit_price","sales","cost","profit"]].notna().all(axis=1) &
                   (df[["quantity","unit_price","sales","cost","profit"]] >= 0).all(axis=1))
flag_disc_missing = df["discount"].isna() & sales_fields_ok
df.loc[flag_disc_missing, "discount"] = 0.0

print("Negative discounts flagged       :", int(flag_disc_neg.sum()))
print("Unusually-high discounts flagged :", int(flag_disc_high.sum()))
print("Missing discounts filled with 0  :", int(flag_disc_missing.sum()))

Negative discounts flagged       : 16
Unusually-high discounts flagged : 8
Missing discounts filled with 0  : 18


## Step 4 - Fill missing region and ship_mode

In [6]:
flag_region_missing = df["region"].isna()
flag_ship_missing   = df["ship_mode"].isna()
df["region"]    = df["region"].fillna("Unknown")
df["ship_mode"] = df["ship_mode"].fillna("Unknown")
print("region filled Unknown   :", int(flag_region_missing.sum()))
print("ship_mode filled Unknown:", int(flag_ship_missing.sum()))

region filled Unknown   : 26
ship_mode filled Unknown: 22


## Step 5 - Handle duplicates

In [7]:
df = df.assign(_neg=flag_disc_neg.values, _high=flag_disc_high.values,
               _dmiss=flag_disc_missing.values, _rmiss=flag_region_missing.values,
               _smiss=flag_ship_missing.values)
n_before  = len(df)
core_cols = [c for c in df.columns if not c.startswith("_")]
exact_dups = df.duplicated(subset=core_cols)
print("Exact duplicate rows removed:", int(exact_dups.sum()))
df = df[~exact_dups].copy()

dup_ids = df["order_id"][df["order_id"].duplicated(keep=False)].unique()
df["_conflict"] = df["order_id"].isin(dup_ids)
print("Conflicting duplicate order_id rows flagged:", int(df["_conflict"].sum()),
      "across", len(dup_ids), "ids")
print("Row count:", n_before, "->", len(df))

Exact duplicate rows removed: 20
Conflicting duplicate order_id rows flagged: 24 across 12 ids
Row count: 932 -> 912


## Step 6 - Date sequence, calculated fields & data_quality_flag

In [8]:
flag_ship_before = (df["ship_date"] < df["order_date"]).fillna(False)
disc_used = df["discount"].where((df["discount"] >= 0) & (df["discount"] <= VALID_MAX), 0.0)
df["calculated_sales"]    = (df["quantity"] * df["unit_price"] * (1 - disc_used)).round(2)
df["calculated_profit"]   = (df["calculated_sales"] - df["cost"]).round(2)
df["profit_margin"]       = (df["calculated_profit"] / df["calculated_sales"]).round(4)
df["shipping_delay_days"] = (df["ship_date"] - df["order_date"]).dt.days
df["order_month"]         = df["order_date"].dt.month
df["order_year"]          = df["order_date"].dt.year

flag_sales_mismatch  = (df["sales"]  - df["calculated_sales"]).abs()  > 1
flag_profit_mismatch = (df["profit"] - df["calculated_profit"]).abs() > 1

issues = pd.DataFrame({
    "region_missing_filled":        df["_rmiss"],
    "ship_mode_missing_filled":     df["_smiss"],
    "discount_missing_filled_zero": df["_dmiss"],
    "discount_negative_invalid":    df["_neg"],
    "discount_unusually_high":      df["_high"],
    "conflicting_duplicate_id":     df["_conflict"],
    "ship_before_order":            flag_ship_before,
    "sales_mismatch":               flag_sales_mismatch,
    "profit_mismatch":              flag_profit_mismatch,
}).fillna(False)
df["data_quality_flag"] = issues.apply(
    lambda r: ";".join([c for c, v in r.items() if v]) or "OK", axis=1)
df["data_quality_flag"].value_counts().head(20)

data_quality_flag
OK                                                                                756
region_missing_filled                                                              23
ship_before_order                                                                  18
ship_mode_missing_filled                                                           16
profit_mismatch                                                                    16
discount_missing_filled_zero;sales_mismatch;profit_mismatch                        13
discount_negative_invalid;sales_mismatch;profit_mismatch                            9
conflicting_duplicate_id                                                            9
conflicting_duplicate_id;sales_mismatch                                             9
sales_mismatch                                                                      8
sales_mismatch;profit_mismatch                                                      7
discount_negative_invalid           

## Step 7 - Save cleaned dataset, quality report & completed-sales summary

In [9]:
clean = df.drop(columns=[c for c in df.columns if c.startswith("_")])
clean.to_excel(f"{DATA}/cleaned_orders.xlsx", index=False)
clean.to_csv(f"{DATA}/cleaned_orders.csv", index=False)

issue_counts = issues.sum().sort_values(ascending=False).astype(int)
issue_counts.to_csv(f"{OUT}/data_quality_report.csv", header=["row_count"])
flagged = int((clean["data_quality_flag"] != "OK").sum())
print("Total rows:", len(clean), "| Flagged:", flagged, "| Fully clean:", len(clean) - flagged)

completed = clean[(clean["order_status"] == "Completed") & (clean["payment_status"] == "Paid")]
summ = completed.groupby("category").agg(
    orders=("order_id", "nunique"),
    sales=("calculated_sales", "sum"),
    profit=("calculated_profit", "sum")).round(2)
summ["margin_%"] = (summ["profit"] / summ["sales"] * 100).round(1)
summ.to_csv(f"{OUT}/completed_sales_by_category.csv")
summ

Total rows: 912 | Flagged: 156 | Fully clean: 756


,orders,sales,profit,margin_%
category,,,,
Furniture,199,1917318.62,554879.08,28.9
Office Supplies,190,1842389.12,512673.51,27.8
Technology,213,2199941.16,655161.29,29.8


## Step 8 - Visual evidence (charts saved to outputs/ and screenshots/)

In [10]:
def save(fig, name):
    fig.tight_layout()
    fig.savefig(f"{OUT}/{name}",   dpi=120, bbox_inches="tight")
    fig.savefig(f"{SHOTS}/{name}", dpi=120, bbox_inches="tight")
    plt.close(fig)

mv = raw[["region","ship_mode","discount"]].isna().sum()
fig, ax = plt.subplots(figsize=(6,4))
mv.plot.bar(ax=ax, color="#c0392b"); ax.set_title("Missing values in raw data"); ax.set_ylabel("rows")
save(fig, "01_missing_values.png")

cmp = pd.DataFrame({"raw": before, "cleaned": after})
fig, ax = plt.subplots(figsize=(8,4))
cmp.plot.bar(ax=ax); ax.set_title("Distinct category values: raw vs cleaned"); ax.set_ylabel("distinct values")
save(fig, "02_category_standardization.png")

fig, ax = plt.subplots(figsize=(8,4))
issue_counts.plot.barh(ax=ax, color="#2c7fb8"); ax.set_title("Data quality issues detected (row counts)")
ax.invert_yaxis()
save(fig, "03_data_quality_flags.png")

fig, ax = plt.subplots(figsize=(7,4))
summ["sales"].plot.bar(ax=ax, color="#27ae60"); ax.set_title("Completed sales by category"); ax.set_ylabel("sales")
save(fig, "04_completed_sales_by_category.png")
print("Charts saved.")

Charts saved.


In [11]:
prev = clean.head(8)[["order_id","order_date","ship_date","region","category",
                       "discount","calculated_sales","calculated_profit","profit_margin","data_quality_flag"]]
fig, ax = plt.subplots(figsize=(15,3)); ax.axis("off")
t = ax.table(cellText=prev.astype(str).values, colLabels=list(prev.columns), loc="center", cellLoc="left")
t.auto_set_font_size(False); t.set_fontsize(7); t.scale(1, 1.5)
ax.set_title("Cleaned data preview", pad=14)
fig.savefig(f"{SHOTS}/05_cleaned_preview.png", dpi=130, bbox_inches="tight"); plt.close(fig)
print("Preview saved.")

Preview saved.


In [12]:
summary = {
    "raw_rows": int(raw.shape[0]), "raw_cols": int(raw.shape[1]),
    "exact_duplicates_removed": int(exact_dups.sum()),
    "final_rows": int(len(clean)),
    "region_filled": int(issues["region_missing_filled"].sum()),
    "ship_mode_filled": int(issues["ship_mode_missing_filled"].sum()),
    "discount_filled_zero": int(issues["discount_missing_filled_zero"].sum()),
    "discount_negative": int(issues["discount_negative_invalid"].sum()),
    "discount_unusually_high": int(issues["discount_unusually_high"].sum()),
    "conflicting_duplicate_id_rows": int(issues["conflicting_duplicate_id"].sum()),
    "ship_before_order": int(issues["ship_before_order"].sum()),
    "sales_mismatch": int(issues["sales_mismatch"].sum()),
    "profit_mismatch": int(issues["profit_mismatch"].sum()),
    "flagged_rows": flagged, "fully_clean_rows": int(len(clean) - flagged),
    "completed_orders": int(completed["order_id"].nunique()),
    "completed_sales_total": float(round(summ["sales"].sum(), 2)),
}
json.dump(summary, open(f"{OUT}/cleaning_summary.json", "w"), indent=2)
summary

{'raw_rows': 932,
 'raw_cols': 21,
 'exact_duplicates_removed': 20,
 'final_rows': 912,
 'region_filled': 25,
 'ship_mode_filled': 21,
 'discount_filled_zero': 18,
 'discount_negative': 15,
 'discount_unusually_high': 8,
 'conflicting_duplicate_id_rows': 24,
 'ship_before_order': 21,
 'sales_mismatch': 56,
 'profit_mismatch': 56,
 'flagged_rows': 156,
 'fully_clean_rows': 756,
 'completed_orders': 602,
 'completed_sales_total': 5959648.9}